# PART 5 📝 Content Layer — TF-IDF + SVD + Content PC

| | |
|---|---|
| **목적** | 형태소 토큰으로 TF-IDF 행렬을 만들고 SVD로 차원 축약하여 Content Layer PC(neg_TFPC2·TFPC1) 생성 |
| **입력** | ESG_passages_tokenized.csv (PART 4.1) · ESG_키워드index_v2.csv (PART 4.2) |
| **출력** | tfidf_sig_pcs.csv (유의 PC 10+neg) · svd_tfidf_optimal.csv (전체 PC) |
| **RQ 연결** | RQ1 Layer 2 Content 피처 생성 — OLS M2·M3b·M4의 neg_TFPC2·TFPC1 |

> **이전 파일**: PART 4.3 `02_텍스트_Intensity.ipynb` — FastText 유사어 확장  
> **다음 파일**: PART 6 `STEP3-1_3_임베딩 (최종).ipynb` — KR-FinBert Semantic Layer

## 5.1 TF-IDF 벡터화

**왜**: 단어 빈도(TF)를 문서 희귀도(IDF)로 가중하면 ESG 공시에서 변별력 있는 단어를 강조할 수 있다.  
**방법**: max_features=500, min_df=2, max_df=0.97 — G 핵심어(이사회 94%)가 max_df=0.8에서 제거됨을 확인하여 0.97로 상향.  
**결정사항**: (368, 500) TF-IDF 행렬 생성.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy import stats

In [2]:
# 토큰화 결과 로드 (STEP2-1 출력)
df = pd.read_csv("data/ESG_passages_tokenized.csv")
print(f"입력: {len(df)}행")

token_counts = df["tokenized"].apply(lambda x: len(str(x).split()))
print(f"평균 토큰 수: {token_counts.mean():.0f}  최소: {token_counts.min()}  최대: {token_counts.max()}")

입력: 368행
평균 토큰 수: 702  최소: 4  최대: 5276


## 📐 STEP 2-3. TF-IDF 벡터화

- `max_features=500`: 상위 500개 토큰
- `min_df=2`: 최소 2개 문서에 등장
- `max_df=0.97`: **STEP2-1 분석에서 G 핵심어(이사회 94%, 독립성 87% 등)가 max_df=0.8에서 제거됨을 확인 → 0.97로 상향**
  - 제거 대상(>97%): 주주(99.5%), 총회(97%) 수준의 완전 보편 단어만 제거
  - 보존 대상: 이사회(94%), 독립성(87%), 위원회(82%), 결의(81%) 등 G 핵심어

In [3]:
vectorizer = TfidfVectorizer(
    max_features=500,
    min_df=2,
    max_df=0.97   # 0.8 → 0.97: G 핵심어(이사회·독립성 등) 보존
)

tfidf_matrix = vectorizer.fit_transform(df["tokenized"])
vocab = vectorizer.get_feature_names_out()

print(f"TF-IDF 행렬: {tfidf_matrix.shape}")

# G 핵심어 보존 여부 검증
g_keys = ['이사회','독립성','사외','감사','위원회','결의','의결권','컴플라이언스']
e_keys = ['탄소','온실가스','에너지','환경','폐기물','재활용']
s_keys = ['안전','임직원','협력사','공급망','인권','노동']

vocab_set = set(vocab)
print("\n=== G 핵심어 TF-IDF 포함 여부 ===")
for w in g_keys:
    mark = "✅" if w in vocab_set else "❌"
    print(f"  {mark} {w}")

print("\n=== E 핵심어 TF-IDF 포함 여부 ===")
for w in e_keys:
    mark = "✅" if w in vocab_set else "❌"
    print(f"  {mark} {w}")

print("\n=== S 핵심어 TF-IDF 포함 여부 ===")
for w in s_keys:
    mark = "✅" if w in vocab_set else "❌"
    print(f"  {mark} {w}")


TF-IDF 행렬: (368, 500)

=== G 핵심어 TF-IDF 포함 여부 ===
  ✅ 이사회
  ✅ 독립성
  ✅ 사외
  ✅ 감사
  ✅ 위원회
  ✅ 결의
  ✅ 의결권
  ❌ 컴플라이언스

=== E 핵심어 TF-IDF 포함 여부 ===
  ✅ 탄소
  ✅ 온실가스
  ✅ 에너지
  ✅ 환경
  ✅ 폐기물
  ✅ 재활용

=== S 핵심어 TF-IDF 포함 여부 ===
  ✅ 안전
  ✅ 임직원
  ✅ 협력사
  ✅ 공급망
  ❌ 인권
  ❌ 노동


## 5.2 SVD 성분 최적화

## 🔢 STEP 2-4. SVD 성분 최적화

- 성분 수 5 → 10 → 15 → 20 → 25 → 30 탐색
- 목표: 누적 설명분산 **60%** 이상

In [4]:
print("=== SVD 성분 수별 누적 분산 ===")
var_table = []
for n in [5, 10, 15, 20, 25, 30, 35, 40, 50]:  # 범위 확장 (max_features=500 기준 60% 달성 위해)
    svd_tmp = TruncatedSVD(n_components=n, random_state=42)
    svd_tmp.fit(tfidf_matrix)
    cum_var = svd_tmp.explained_variance_ratio_.sum()
    var_table.append({"n_components": n, "cumulative_variance": round(cum_var, 4)})
    mark = " ← 목표 달성" if cum_var >= 0.60 else ""
    print(f"  n={n:2d}: {cum_var:.1%}{mark}")

var_df = pd.DataFrame(var_table)

optimal_n = var_df[var_df["cumulative_variance"] >= 0.60]["n_components"].min()
if pd.isna(optimal_n):
    optimal_n = 50
    print(f"⚠️  60% 미달 — n={optimal_n} 사용")
else:
    print(f"\n✅  최적 성분 수: n={int(optimal_n)}")
optimal_n = int(optimal_n)


=== SVD 성분 수별 누적 분산 ===
  n= 5: 19.7%
  n=10: 30.3%
  n=15: 38.1%
  n=20: 44.5%
  n=25: 50.0%
  n=30: 54.8%
  n=35: 58.9%
  n=40: 62.7% ← 목표 달성
  n=50: 69.1% ← 목표 달성

✅  최적 성분 수: n=40


In [5]:
svd = TruncatedSVD(n_components=optimal_n, random_state=42)
tfpc = svd.fit_transform(tfidf_matrix)

exp_var = svd.explained_variance_ratio_
var_result = pd.DataFrame({
    "component":          [f"TFPC{i+1}" for i in range(optimal_n)],
    "explained_variance": exp_var.round(4),
    "cumulative_variance": exp_var.cumsum().round(4)
})

print(f"총 설명분산: {exp_var.sum():.1%}")
print(var_result.to_string(index=False))

총 설명분산: 62.7%
component  explained_variance  cumulative_variance
    TFPC1              0.0141               0.0141
    TFPC2              0.0779               0.0920
    TFPC3              0.0453               0.1373
    TFPC4              0.0316               0.1689
    TFPC5              0.0279               0.1968
    TFPC6              0.0254               0.2221
    TFPC7              0.0218               0.2439
    TFPC8              0.0216               0.2655
    TFPC9              0.0192               0.2847
   TFPC10              0.0179               0.3026
   TFPC11              0.0172               0.3199
   TFPC12              0.0162               0.3361
   TFPC13              0.0157               0.3518
   TFPC14              0.0155               0.3673
   TFPC15              0.0142               0.3815
   TFPC16              0.0141               0.3956
   TFPC17              0.0133               0.4089
   TFPC18              0.0129               0.4218
   TFPC19        

In [6]:
tfpc_df = pd.DataFrame(tfpc, columns=[f"TFPC{i+1}" for i in range(optimal_n)])
tfpc_df.insert(0, "fiscal_year", df["fiscal_year"].values)
tfpc_df.insert(0, "stock_code",  df["stock_code"].values)

tfpc_df.to_csv("svd_tfidf_optimal.csv", index=False, encoding="utf-8-sig")
print(f"저장: svd_tfidf_optimal.csv  {tfpc_df.shape}")

var_result.to_csv("svd_explained_variance.csv", index=False, encoding="utf-8-sig")
print(f"저장: svd_explained_variance.csv")

저장: svd_tfidf_optimal.csv  (368, 42)
저장: svd_explained_variance.csv


## 5.3 PC-ESG 상관분석

## 📊 STEP 2-6. PC-ESG 상관분석

- **Pearson**: 선형 상관
- **Spearman**: 비선형(순위) 상관
- TFPCi × E_seed / S_seed / G_seed

In [7]:
# Seed 점수 로드 (STEP2-2 출력)
seed_df = pd.read_csv("data/ESG_키워드index_v2.csv")
print(f"seed 로드: {seed_df.shape}")

combined = tfpc_df.merge(
    seed_df[["stock_code","fiscal_year","E_seed","S_seed","G_seed"]],
    on=["stock_code","fiscal_year"], how="left"
)
print(f"병합 결과: {combined.shape}")

seed 로드: (368, 31)
병합 결과: (368, 45)


In [8]:
pc_cols  = [f"TFPC{i+1}" for i in range(optimal_n)]
dim_cols = ["E_seed", "S_seed", "G_seed"]

rows = []
for pc in pc_cols:
    for dim in dim_cols:
        x = combined[pc].values
        y = combined[dim].values
        r_p, p_p = stats.pearsonr(x, y)
        r_s, p_s = stats.spearmanr(x, y)
        rows.append({
            "PC": pc, "dimension": dim.replace("_seed",""),
            "pearson_r": round(r_p,4), "pearson_p": round(p_p,4),
            "spearman_r": round(r_s,4), "spearman_p": round(p_s,4),
        })

corr_df = pd.DataFrame(rows)

sig = corr_df[(corr_df["pearson_p"] < 0.05) & (corr_df["pearson_r"].abs() >= 0.20)]
print(f"=== 유의한 Pearson 상관 (|r|≥0.2, p<0.05): {len(sig)}건 ===")
print(sig.sort_values("pearson_r", key=abs, ascending=False).to_string(index=False))

=== 유의한 Pearson 상관 (|r|≥0.2, p<0.05): 11건 ===
    PC dimension  pearson_r  pearson_p  spearman_r  spearman_p
 TFPC2         E    -0.6623     0.0000     -0.8053         0.0
 TFPC1         G     0.5384     0.0000      0.5487         0.0
 TFPC2         S    -0.4183     0.0000     -0.3978         0.0
 TFPC4         G    -0.3579     0.0000     -0.4046         0.0
 TFPC3         G    -0.3342     0.0000     -0.2751         0.0
 TFPC1         E     0.3024     0.0000      0.5099         0.0
 TFPC7         G    -0.2800     0.0000     -0.2763         0.0
TFPC20         G     0.2307     0.0000      0.2545         0.0
 TFPC1         S     0.2295     0.0000      0.3562         0.0
TFPC15         S    -0.2065     0.0001     -0.2203         0.0
 TFPC4         S    -0.2054     0.0001     -0.2806         0.0


In [9]:
corr_df.to_csv("PC_ESG_correlation_content.csv", index=False, encoding="utf-8-sig")
print("저장: PC_ESG_correlation_content.csv")

# PC 해석표 작성
summary_rows = []
for pc in pc_cols:
    sub  = corr_df[corr_df["PC"] == pc].copy()
    best = sub.loc[sub["pearson_r"].abs().idxmax()]
    summary_rows.append({
        "PC": pc,
        "best_dim": best["dimension"],
        "pearson_r": best["pearson_r"],
        "pearson_p": best["pearson_p"],
        "cumulative_variance": var_result.loc[var_result["component"]==pc, "cumulative_variance"].values[0]
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

with open("PC해석표_Content.md", "w", encoding="utf-8") as f:
    f.write("# PC 해석표 — Content Layer (TF-IDF SVD)\n\n")
    f.write(f"- 최적 성분 수: {optimal_n}\n")
    f.write(f"- 총 설명분산: {exp_var.sum():.1%}\n\n")
    f.write("| PC | 주요 연관 차원 | Pearson r | p-value | 누적 분산 |\n")
    f.write("|----|-----------|-----------|---------|-----------  |\n")
    for _, row in summary.iterrows():
        sig_mark = "*" if row["pearson_p"] < 0.05 else ""
        f.write(f"| {row['PC']} | {row['best_dim']}{sig_mark} | {row['pearson_r']:.3f} | {row['pearson_p']:.3f} | {row['cumulative_variance']:.1%} |\n")
    f.write("\n*p < 0.05\n")

print("저장: PC해석표_Content.md")

저장: PC_ESG_correlation_content.csv
    PC best_dim  pearson_r  pearson_p  cumulative_variance
 TFPC1        G     0.5384     0.0000               0.0141
 TFPC2        E    -0.6623     0.0000               0.0920
 TFPC3        G    -0.3342     0.0000               0.1373
 TFPC4        G    -0.3579     0.0000               0.1689
 TFPC5        E     0.1312     0.0118               0.1968
 TFPC6        S     0.1154     0.0269               0.2221
 TFPC7        G    -0.2800     0.0000               0.2439
 TFPC8        E     0.1232     0.0181               0.2655
 TFPC9        E    -0.0552     0.2911               0.2847
TFPC10        S    -0.1464     0.0049               0.3026
TFPC11        E     0.1251     0.0164               0.3199
TFPC12        S     0.1606     0.0020               0.3361
TFPC13        S    -0.0636     0.2237               0.3518
TFPC14        S     0.0578     0.2685               0.3673
TFPC15        S    -0.2065     0.0001               0.3815
TFPC16        S     0

저장: PC해석표_Content.md

## 5.4 유의 PC 선별 + neg_TFPC2 추가

## 🔧 보완: 유의 PC 선별 + neg_TFPC2 추가 (STEP3과 동일 방식)

In [10]:
# ── 1. 유의 PC 선별 (|r|≥0.15, p<0.05) ─────────────────────
sig = corr_df[(corr_df['pearson_p'] < 0.05) & (corr_df['pearson_r'].abs() >= 0.15)]
sig_pcs = sorted(sig['PC'].unique(), key=lambda x: int(x.replace('TFPC','')))
print(f"유의 PC ({len(sig_pcs)}개): {sig_pcs}")

# ── 2. neg_TFPC2 추가 (E/S와 음의 상관 → 방향 반전) ─────────
# 근거: TFPC2 × E = -0.662 (Pearson), -0.805 (Spearman)
#       TFPC2가 낮을수록 E·S 내용 강함 → -TFPC2가 E/S 강도 축
tfpc_df['neg_TFPC2'] = -tfpc_df['TFPC2']

# 검증
merged = tfpc_df.merge(seed_df[['stock_code','fiscal_year','E_seed','S_seed','G_seed']],
                        on=['stock_code','fiscal_year'], how='left')
for dim in ['E','S','G']:
    r_p, p_p = stats.pearsonr(merged['neg_TFPC2'], merged[f'{dim}_seed'])
    r_s, _   = stats.spearmanr(merged['neg_TFPC2'], merged[f'{dim}_seed'])
    print(f"neg_TFPC2 × {dim}_seed: Pearson={r_p:.3f}(p={p_p:.4f}), Spearman={r_s:.3f}")

# ── 3. 유의 PC + neg_TFPC2 저장 ──────────────────────────────
key_cols = ['stock_code','fiscal_year'] + sig_pcs + ['neg_TFPC2']
sig_pc_df = tfpc_df[key_cols].copy()
sig_pc_df.to_csv("tfidf_sig_pcs.csv", index=False, encoding='utf-8-sig')
print(f"\n저장: tfidf_sig_pcs.csv  {sig_pc_df.shape}")
print(f"컬럼: {list(sig_pc_df.columns)}")


유의 PC (10개): ['TFPC1', 'TFPC2', 'TFPC3', 'TFPC4', 'TFPC7', 'TFPC12', 'TFPC15', 'TFPC17', 'TFPC20', 'TFPC26']
neg_TFPC2 × E_seed: Pearson=0.662(p=0.0000), Spearman=0.805
neg_TFPC2 × S_seed: Pearson=0.418(p=0.0000), Spearman=0.398
neg_TFPC2 × G_seed: Pearson=-0.040(p=0.4485), Spearman=-0.060

저장: tfidf_sig_pcs.csv  (368, 13)
컬럼: ['stock_code', 'fiscal_year', 'TFPC1', 'TFPC2', 'TFPC3', 'TFPC4', 'TFPC7', 'TFPC12', 'TFPC15', 'TFPC17', 'TFPC20', 'TFPC26', 'neg_TFPC2']


### 5.4.1 Content PC 성분 해석 (상위 로딩 Top10)

## 🔍 P2-1. Content PC 성분 해석 (상위 로딩 단어 Top10)

**막는 비판**: "PC가 무엇을 의미하는지 알 수 없다"

In [ ]:
import pandas as pd

df = pd.read_csv('pc_loading_top10.csv')

# 수작업 라벨 (로딩 단어 기반 해석)
labels = {
    'TFPC1':  'G·E 혼합축 (이사회·감사 + 에너지·환경)',
    'TFPC2':  'E/S 강도축 — neg_TFPC2 채택 (E 탄소·배출 vs G 이사회)',
    'TFPC3':  'G 선임 vs 재무리스크 분리축',
    'TFPC4':  'S 안전·산업 vs E·G 리스크 분리축',
    'TFPC7':  'G 금융리스크 vs E 에너지 분리축',
    'TFPC12': 'S 자동차·안전·수소 업종 특화축',
    'TFPC15': 'G 의결권·투표 vs S 교육·보건 분리축',
    'TFPC17': 'G 준법·컴플라이언스 vs E 인증·규정 분리축',
    'TFPC20': 'S 정보보안·개인정보 vs E 소재 분리축',
    'TFPC26': 'S 교육·개발 vs G 준법·선임 분리축',
}
df['해석 라벨'] = df['PC'].map(labels)

print('=== Content Layer PC 해석표 (상위 로딩 단어 기반) ===')
for _, row in df.iterrows():
    print(f'\n[{row["PC"]}] {row["해석 라벨"]}')
    print(f'  (+) {row["Top10_양(+)"]}')
    print(f'  (-) {row["Top10_음(-)"]}')

print('\n=== neg_TFPC2 채택 근거 ===')
print('TFPC2 음(-) 방향: 에너지·환경·폐기물·탄소·온실가스')
print('→ 음의 방향이 E/S 강도를 대표하므로 neg_TFPC2 = -1×TFPC2로 방향 반전')
print('→ neg_TFPC2 × E등급 Pearson r = +0.662 (방향 보정 후)')


### 5.4.2 LDA 토픽 수 선택 근거 (Coherence Score)

## 📊 P2-2. LDA 토픽 수 선택 근거 (Coherence Score)

**막는 비판**: "왜 k=6인가? k가 달라지면 결과도 달라지지 않는가?"

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 결과 로드
coh_df = pd.read_csv('lda_coherence_by_k.csv')

print('=== LDA k별 Coherence Score (c_v) ===')
print(coh_df.to_string(index=False))
best_k = coh_df.loc[coh_df['coherence_cv'].idxmax(), 'k']
best_cv = coh_df['coherence_cv'].max()
print(f'\n→ 최적 k = {best_k}  (c_v = {best_cv:.4f}) ✅')
print('→ k=6 선택 근거: Coherence Score 전체 k 중 최고값')

# 시각화
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(coh_df['k'], coh_df['coherence_cv'], 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axvline(x=best_k, color='tomato', linestyle='--', linewidth=1.5, label=f'k={best_k} (최적)')
ax.scatter([best_k], [best_cv], color='tomato', zorder=5, s=80)
ax.set_xlabel('토픽 수 (k)', fontsize=12)
ax.set_ylabel('Coherence Score (c_v)', fontsize=12)
ax.set_title('LDA 토픽 수별 Coherence Score', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(coh_df['k'])
plt.tight_layout()
plt.savefig('lda_coherence_plot.png', dpi=150, bbox_inches='tight')
plt.close()
print('\n저장: lda_coherence_plot.png')

## 📋 결과 검증 및 평가 (최종)

### ✅ 실행 결과 요약

| 지표 | 초기 (max_df=0.8) | 보완 후 (max_df=0.97, n=40) |
|------|-------------------|--------------------------|
| TF-IDF 행렬 | (368, 500) | (368, 500) |
| 최적 성분 수 | 30 (54.8%, 목표 미달) | **40 (62.7%, 목표 달성)** |
| G 핵심어 포함 | 이사회·독립성 등 ❌ 제거 | 이사회·독립성 등 ✅ 포함 |
| 유의 PC 수 | — | **10개** |

### ✅ max_df=0.97의 효과 — G 핵심어 보존

| 키워드 | 문서 출현율 | max_df=0.8 | max_df=0.97 |
|--------|------------|-----------|------------|
| 이사회 | 94.0% | ❌ 제거 | ✅ 포함 |
| 독립성 | 86.7% | ❌ 제거 | ✅ 포함 |
| 위원회 | 82.3% | ❌ 제거 | ✅ 포함 |
| 결의   | 81.0% | ❌ 제거 | ✅ 포함 |
| 환경   | 76.4% | ✅ 포함 | ✅ 포함 |

> `컴플라이언스`, `인권`, `노동`은 max_features=500 컷오프로 미포함 — 빈도 자체가 낮아 TF-IDF 기여도 미미

### ✅ 주요 PC-ESG 상관 (Content Layer)

| PC | 주요 차원 | Pearson r | Spearman r | 해석 |
|----|---------|-----------|------------|------|
| TFPC2 | E | -0.662 | -0.805 | E·S 핵심 축 (음수) → neg_TFPC2로 방향 통일 |
| TFPC1 | G | +0.538 | +0.549 | 전반적 ESG 공시 강도 축 |
| TFPC4 | G | -0.358 | -0.405 | G 보조 축 |
| TFPC3 | G | -0.334 | -0.275 | G 보조 축 |
| TFPC7 | G | -0.280 | -0.276 | G 보조 축 |

**neg_TFPC2 (E/S 공시 강도 축):**
- neg_TFPC2 × E_seed: **Pearson=+0.662**, Spearman=+0.805
- neg_TFPC2 × S_seed: Pearson=+0.418, Spearman=+0.398
- neg_TFPC2 × G_seed: Pearson=-0.040 (p=0.45) → **G와 독립적** — SEPC의 neg_SEPC1과 동일 패턴

### ⚠️ 잔존 한계

**1. SVD 설명분산의 구조적 한계**
- TF-IDF는 희소 행렬 → SVD 설명분산이 낮음 (n=40에서 62.7%)
- 이는 TF-IDF+SVD (LSA)의 고유 특성 — 의미 유사도보다 단어 빈도 기반
- 실제 ESG 상관(TFPC2×E=-0.662)은 강하므로 분석적 유효성은 있음

**2. G 신호 분산**
- G와 유의한 상관: TFPC1(+0.538), TFPC3(-0.334), TFPC4(-0.358), TFPC7(-0.280) — 4개 PC에 분산
- Content Layer도 Semantic Layer와 동일하게 G 단일 축 없음
- PHASE5에서: G 관련 TFPC 합산 또는 개별 투입 비교

**3. Content vs Semantic 비교**
| 지표 | Content (TF-IDF) | Semantic (KoSimCSE) |
|------|-----------------|---------------------|
| E 최강 상관 | TFPC2×E = -0.662 | neg_SEPC1×E = +0.595 |
| Spearman(E) | -0.805 | +0.786 |
| G 주축 | TFPC1×G = +0.538 | SEPC7×G = +0.319 |
| G 분산 | 4개 PC | 5개 PC |

→ **E 차원: Content Layer가 Semantic보다 강한 상관** (TF-IDF의 단어 빈도가 E 공시량 측정에 유리)  
→ **G 차원: Content Layer가 더 명확한 주축** (TFPC1이 G를 0.538로 포착)

### 📌 PHASE5에 전달할 사항
- **E/S 피처**: `neg_TFPC2` (Content) + `neg_SEPC1` (Semantic) → 두 레이어 비교
- **G 피처**: `TFPC1` (Content, 주축) + `SEPC7` (Semantic, 주축)
- **유의 PC CSV**: `tfidf_sig_pcs.csv` (368×13) → 직접 사용
- **Spearman > Pearson**: E 차원 비선형 관계 강함 → ordered logit 권장
- **low_token_flag**: sensitivity check에서 5개 문서 제외 후 비교

## 📋 P2-4. 분석 역할 요약 (프레임 내 위치)

### 이 노트북의 역할
- **목적**: TF-IDF(500 features) + SVD 차원 축약 → Content Layer PC 생성
- **입력**: ESG_passages_tokenized.csv (Kiwi 형태소, 368행)
- **주요 출력**: tfidf_sig_pcs.csv (유의 PC 10개), lda_topics_k6.csv (LDA k=6)

### 핵심 결정
| 결정 | 선택 | 근거 |
|------|------|------|
| max_df | 0.97 | 0.8이면 이사회(94%)·독립성(87%) 제거됨 |
| SVD 성분 수 | 40 | 62.7% 설명분산 (60% 목표 달성) |
| LDA k | 6 | Coherence Score c_v=0.480 (k=3~10 중 최고) |

### 통합 모델 기여
| Feature | ESG R² 기여 |
|---------|------------|
| neg_TFPC2 (E/S축) | Spearman ρ=-0.805 (E등급) |
| TFPC1 (G축) | Pearson r=+0.538 (G등급) |
| Full + Content → R² | 0.182 → 0.249 (+6.7%p) |

---
✅ **PART 5 완료** — Content Layer

생성 파일:
- `tfidf_sig_pcs.csv` — 유의 PC (10개 + neg_TFPC2)
- `svd_tfidf_optimal.csv` — 전체 SVD PC

핵심 피처: `neg_TFPC2`(G 공시 부정적 패턴 역전) · `TFPC1`(E/S 긍정적 공시 패턴)

> **다음 파일**: PART 6 `STEP3-1_3_임베딩 (최종).ipynb` — KR-FinBert 임베딩 Semantic Layer